# CP3-05 查找错误：用 StateSnapshot 考古失败现场

本 Notebook 是错误三部曲的第二步。它不再重新 `invoke` 图，而是读取 CP3-04 使用同一 `thread_id` 写入的检查点历史，从 `StateSnapshot.tasks` 中定位具体失败节点。

## 执行顺序与前提

请先执行 `04_error.ipynb`。如果历史为空，通常意味着 PostgreSQL 连接串不同、数据库尚未建表，或 CP3-04 没有运行过。

重点字段：`values` 是已提交状态，`next` 是下一个超步准备执行的节点，`metadata.step` 表示超步编号，`tasks` 中的 `result` 与 `error` 用于判断每个任务是否成功。


In [ ]:
import os
from typing import TypedDict

from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph

# 本节只读取历史，不调用模型；仍需构建同形状的图才能调用 get_state_history。
load_dotenv(override=True)
DB_URL = os.getenv('LANGGRAPH_DB_URL')
if not DB_URL:
    raise RuntimeError('缺少 LANGGRAPH_DB_URL，请配置 PostgreSQL 连接串。')

class OverAllState(TypedDict, total=False):
    topic: str
    poem: str
    joke: str
    final_output: str

class InputState(TypedDict):
    topic: str

class OutputState(TypedDict):
    final_output: str

topics = ['布偶猫', '狸花猫', '金渐层']
topic_index = 0

def node_change_topic(state: InputState) -> OverAllState:
    global topic_index
    sub_topic = topics[topic_index]
    topic_index = (topic_index + 1) % len(topics)
    return {'topic': f'{state["topic"]}:{sub_topic}'}

def node_poem(state: OverAllState) -> OverAllState:
    # 本节只读历史，误调用图时给出明确提示而不是引用未初始化的模型。
    raise RuntimeError('CP3-05 只用于读取检查点，不应重新执行 node_poem。')

def node_joke(state: OverAllState) -> OverAllState:
    # 保留 CP3-04 的故障节点定义；本节不会调用它。
    raise RuntimeError('人为抛异常：CP3-04 的失败现场')

def node_output(state: OverAllState) -> OutputState:
    return {'final_output': f'关于{state["topic"]}的七言绝句:{state["poem"]}' + chr(10) + f'笑话:{state["joke"]}'}

builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)
builder.add_node('node_change_topic', node_change_topic)
builder.add_node('node_poem', node_poem)
builder.add_node('node_joke', node_joke)
builder.add_node('node_output', node_output)
builder.add_edge(START, 'node_change_topic')
builder.add_edge('node_change_topic', 'node_poem')
builder.add_edge('node_change_topic', 'node_joke')
builder.add_edge('node_poem', 'node_output')
builder.add_edge('node_joke', 'node_output')
builder.add_edge('node_output', END)

from langgraph.checkpoint.postgres import PostgresSaver
THREAD_ID = os.getenv('CP3_ERROR_THREAD_ID', 'chapter03-05')
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)
    config = {'configurable': {'thread_id': THREAD_ID}}
    state_history = list(graph.get_state_history(config=config))
    if not state_history:
        raise RuntimeError('没有找到检查点，请先按顺序执行 CP3-04_error.ipynb。')

    # 不直接打印完整快照，避免 LLM 长文本和内部元数据淹没故障线索。
    for snapshot in state_history:
        task_summary = [
            {
                'name': task.name,
                'error': repr(task.error) if task.error else None,
                'has_result': task.result is not None,
            }
            for task in snapshot.tasks
        ]
        print({
            'step': snapshot.metadata.get('step'),
            'next': snapshot.next,
            'value_keys': sorted(snapshot.values.keys()),
            'tasks': task_summary,
        })


## 如何读出这次错误

最近的 step 1 快照中，`node_poem` 应有 `has_result=True`，而 `node_joke` 应带有 `RuntimeError` 文本。`node_output` 没有出现在待执行列表中，说明汇总节点尚未运行。

与只看终端 traceback 相比，检查点还保留了“哪些工作已提交、哪些任务未完成、从哪里继续”的信息。下一节会移除故障注入，并从相同的检查点恢复。
